# Предсказание цен на жилье в городе Эймс, штат Айова, США

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [38]:
from config import config

In [39]:
train_df = pd.read_csv(config.train_data_path, na_values=['?', '-', ' ', 'N/A', 'NA', ''])
test_df = pd.read_csv(config.test_data_path, na_values=['?', '-', ' ', 'N/A', 'NA', ''])

## Описание признаков датасета

**Идентификация и целевая переменная**
- `Id` — уникальный идентификатор дома
- `SalePrice` — цена продажи (целевая переменная)

**Участок и зонирование**
- `MSSubClass` — класс жилья (тип строения)
- `MSZoning` — классификация зонирования
- `LotFrontage` — длина фасада участка вдоль улицы (футы)
- `LotArea` — площадь участка (кв. футы)
- `Street` — тип подъездной дороги (гравий/асфальт)
- `Alley` — тип доступа через переулок
- `LotShape` — форма участка
- `LandContour` — ровность участка
- `Utilities` — доступные коммуникации
- `LotConfig` — конфигурация участка
- `LandSlope` — уклон участка
- `Neighborhood` — район города
- `Condition1` — близость к дорогам/ж/д и пр.
- `Condition2` — вторичная близость к объектам

**Тип и стиль дома**
- `BldgType` — тип жилого строения
- `HouseStyle` — стиль дома

**Качество и возраст**
- `OverallQual` — общее качество материалов и отделки (1–10)
- `OverallCond` — общее состояние дома (1–10)
- `YearBuilt` — год постройки
- `YearRemodAdd` — год последней реконструкции

**Крыша**
- `RoofStyle` — тип крыши
- `RoofMatl` — материал кровли

**Экстерьер**
- `Exterior1st` — основной материал наружной отделки
- `Exterior2nd` — доп. материал наружной отделки
- `MasVnrType` — тип кирпичной/каменной облицовки
- `MasVnrArea` — площадь облицовки (кв. футы)
- `ExterQual` — качество материалов экстерьера
- `ExterCond` — текущее состояние экстерьера

**Фундамент и подвал**
- `Foundation` — тип фундамента
- `BsmtQual` — высота потолков в подвале
- `BsmtCond` — общее состояние подвала
- `BsmtExposure` — уровень цокольного выхода
- `BsmtFinType1` — качество отделки подвала (тип 1)
- `BsmtFinSF1` — площадь отделки подвала (тип 1)
- `BsmtFinType2` — качество отделки подвала (тип 2)
- `BsmtFinSF2` — площадь отделки подвала (тип 2)
- `BsmtUnfSF` — неотделанная площадь подвала
- `TotalBsmtSF` — общая площадь подвала
- `BsmtFullBath` — полные ванные в подвале
- `BsmtHalfBath` — полуванные в подвале

**Отопление и коммуникации**
- `Heating` — тип отопления
- `HeatingQC` — качество и состояние отопления
- `CentralAir` — наличие центрального кондиционирования
- `Electrical` — тип электропроводки

**Планировка**
- `1stFlrSF` — площадь 1-го этажа
- `2ndFlrSF` — площадь 2-го этажа
- `LowQualFinSF` — площадь отделки низкого качества
- `GrLivArea` — жилая площадь над землёй (кв. футы)
- `FullBath` — полные ванные над землёй
- `HalfBath` — полуванные над землёй
- `BedroomAbvGr` — спальни над землёй
- `KitchenAbvGr` — кухни над землёй
- `KitchenQual` — качество кухни
- `TotRmsAbvGrd` — всего комнат над землёй
- `Functional` — функциональность дома

**Камины**
- `Fireplaces` — количество каминов
- `FireplaceQu` — качество камина

**Гараж**
- `GarageType` — тип гаража
- `GarageYrBlt` — год постройки гаража
- `GarageFinish` — внутренняя отделка гаража
- `GarageCars` — вместимость (машиномест)
- `GarageArea` — площадь гаража (кв. футы)
- `GarageQual` — качество гаража
- `GarageCond` — состояние гаража

**Двор и внешние элементы**
- `PavedDrive` — наличие мощёной дорожки
- `WoodDeckSF` — площадь деревянной террасы
- `OpenPorchSF` — площадь открытого крыльца
- `EnclosedPorch` — площадь закрытого крыльца
- `3SsnPorch` — площадь трёхсезонного крыльца
- `ScreenPorch` — площадь крыльца с экраном
- `PoolArea` — площадь бассейна
- `PoolQC` — качество бассейна
- `Fence` — качество забора
- `MiscFeature` — прочие особенности (лифт, сарай и т.д.)
- `MiscVal` — стоимость прочих особенностей ($)

**Продажа**
- `MoSold` — месяц продажи (1–12)
- `YrSold` — год продажи
- `SaleType` — тип продажи
- `SaleCondition` — условия продажи

## Анализ данных

In [40]:
train_df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [41]:
print("Размерность train_df:", train_df.shape)
print("Размерность test_df:", test_df.shape)

Размерность train_df: (1460, 81)
Размерность test_df: (1459, 80)


Проверяем совпадают ли признаки у тренировочной и тестовой выборки

In [42]:
train_num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
train_cat_cols = train_df.select_dtypes(include=['object', 'category', 'bool', 'str']).columns.tolist()

test_num_cols = test_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
test_cat_cols = test_df.select_dtypes(include=['object', 'category', 'bool', 'str']).columns.tolist()


exclude_train = ['Id', 'SalePrice', 'MSSubClass']
train_num_cols = [col for col in train_num_cols if col not in exclude_train]

exclude_test = ['Id', 'MSSubClass']
test_num_cols = [col for col in test_num_cols if col not in exclude_test]

train_cat_cols.append('MSSubClass')
test_cat_cols.append('MSSubClass')

print(f'Количество различающихся числовых признаков: {len(set(train_num_cols) - set(test_num_cols))}')
print(f'Количество различающихся категориальных признаков: {len(set(train_cat_cols) - set(test_cat_cols))}')


Количество различающихся числовых признаков: 0
Количество различающихся категориальных признаков: 0


Вывод: отличий в признаках между тренировочным и тестовым датасетом нет

In [43]:
num_features = list(train_num_cols)
cat_features = list(train_cat_cols)
features = num_features + cat_features
target_feature = 'SalePrice'

In [44]:
print(f"{len(num_features)} числовых признаков")
print(f"{len(cat_features)} категориальных признаков")

35 числовых признаков
44 категориальных признаков


In [45]:
if config.eda_condition == True:
    train_df.describe(include=['int64', 'float64']).T
else: 
    print(f"Описание числовых признаков")

Описание числовых признаков


Изучим пропущенные значения в признаках

In [46]:
if config.eda_condition == True:
    missings = train_df.isna().sum()
    missings[missings > 0]
else: 
    print(f"Описание пропущенных")

Описание пропущенных


In [47]:
num_train_df = train_df[num_features]
cat_train_df = train_df[cat_features]

#### Анализ числовых признаков

In [48]:
if config.eda_condition == True:
    num_train_df.hist(figsize=(20, 15), bins=50)
    plt.tight_layout()
    plt.show()
else: 
    print(f"Гистограммы числовых признаков")

Гистограммы числовых признаков


In [49]:
if config.eda_condition == True:
    corr = num_train_df.corr()

    plt.figure(figsize=(14, 12))
    sns.heatmap(corr, annot=False, fmt='.2f', cmap='coolwarm', square=True)
    plt.title('Корреляционная матрица')
    plt.show()
else: 
    print(f"Матрица корреляций числовых признаков")

Матрица корреляций числовых признаков


In [50]:
from src.utils import visualize_numerical_features

if config.eda_condition == True:
    visualize_numerical_features(train_df, num_features)
else: 
    print(f"Боксплоты числовых признаков")

Боксплоты числовых признаков


#### Анализ категориальных признаков

In [51]:
from src.utils import visualize_categorical_features

if config.eda_condition == True:
    visualize_categorical_features(train_df, cat_features)
else: 
    print(f"Распределение категориальных признаков")

Распределение категориальных признаков


## Обработка данных

Разделим данные на тренирововчные и тестовые выборки

In [52]:
from sklearn.model_selection import train_test_split

X = train_df[features]
y = train_df[target_feature]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [53]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1022, 79)
y_train shape: (1022,)
X_test shape: (438, 79)
y_test shape: (438,)


In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from src.utils import NumericalImputer, NumericalScaler
from src.utils import CategoricalImputer, CategoricalEncoder

num_imputer = NumericalImputer(num_cols=num_features, 
                               impute_strategy='simple_imputer', 
                               imputer_params={'strategy': 'mean'}
                            )

num_scaler = NumericalScaler(num_cols=num_features,
                             scaler_type='standard_scaler',
                             scaler_params={'copy': True, 'with_mean': True, 'with_std': True}
                            )

cat_imputer = CategoricalImputer(cat_cols=cat_features, 
                                 filler_strategy='mode',                                  
                                )

cat_encoder = CategoricalEncoder(cat_cols=cat_features,
                                 encoder_strategy='one_hot_encoding',
                                 encoder_params={'drop': 'if_binary'}
                                )

num_pipeline = Pipeline([('imputer', num_imputer),
                         ('scaler', num_scaler)])

cat_pipeline = Pipeline([('imputer', cat_imputer),
                         ('encoder', cat_encoder)])

preprocessor = ColumnTransformer([('num', num_pipeline, num_features),
                                  ('cat', cat_pipeline, cat_features)])                                 


In [55]:
from src.utils.utils import check_missing_values

preprocessor.fit(X_train)

all_features = preprocessor.get_feature_names_out()

X_train_preprocessed = pd.DataFrame(preprocessor.transform(X_train), 
                                    columns=list(all_features))
if sum(check_missing_values(X_train_preprocessed)) > 0:
    print("В данных есть пропущенные значения:")   
else:
    print("Пропущенных значений нет") 

check_missing_values(X_train_preprocessed)
print(X_train_preprocessed.shape)

(1022, 35)
(1022, 35)
Пропущенных значений нет
(1022, 292)


#### Испытания RandomForestRegressor

In [56]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA

rf_test_model = RandomForestRegressor(
    n_estimators=200,           
    max_depth=15,               
    min_samples_split=10,       
    min_samples_leaf=5,         
    max_features='sqrt',        
    bootstrap=True,             
    random_state=42,
    n_jobs=-1                   
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', rf_test_model)
])

pipeline.fit(X_train, y_train)

train_predictions = pipeline.predict(X_train)
test_predictions = pipeline.predict(X_test)

(1022, 35)
(1022, 35)
(438, 35)


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [57]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.utils import show_metrics

print("=== МЕТРИКИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===")
show_metrics(train_predictions, y_train)

print("----------------------------------")

print("=== МЕТРИКИ НА ТЕСТОВЫХ ДАННЫХ ===")
show_metrics(test_predictions, y_test)

=== МЕТРИКИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $15,561
RELATIVE MAE ERROR:  0.086
RMSE: $28,360
RELATIVE RMSE ERROR:  0.156
R²:   0.7856
----------------------------------
=== МЕТРИКИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $19,068
RELATIVE MAE ERROR:  0.106
RMSE: $34,140
RELATIVE RMSE ERROR:  0.19
R²:   0.6924


### Подбор гиперпараметров для RandomForest

In [58]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
import joblib

Гиперпараметры для RandomForest

In [59]:
rf_model = RandomForestRegressor(random_state=42)

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', rf_model)
])

rf_param_grid = {
    'model': rf_model,
    'model__n_estimators': [50, 100, 200],        
    'model__max_depth': [10, 20, None],           
    'model__min_samples_split': [2, 5],           
    'model__min_samples_leaf': [1, 2],            
    'pca__n_components': [0.7, 0.8, None]        
}

rf_param_distribution = {    
    'model__n_estimators': randint(100, 400),   
    'model__max_depth': randint(10, 20),   
    'model__min_samples_split': randint(5, 20),
    'model__min_samples_leaf': randint(2, 10),
    'model__max_features': [0.5, 0.7, 0.9],
    'model__bootstrap': [True, False]
}

grid_search_rf = GridSearchCV(
                       estimator=rf_pipeline,
                       param_grid=rf_param_grid,
                       cv=5,
                       scoring='neg_mean_absolute_error',
                       n_jobs=-1,
                       verbose=3)

random_search_rf = RandomizedSearchCV(
                        estimator=rf_pipeline,
                        param_distributions=rf_param_distribution,
                        n_iter=100,
                        cv=5,
                        scoring='neg_mean_absolute_error',
                        n_jobs=-1,                      
                        verbose=2,                      
                        random_state=42)

# grid_search_rf.fit(X_train, y_train)

if (config.model_storage_path / 'rf_best_model.pkl').exists():
    rf_best_model = joblib.load(config.model_storage_path / 'rf_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    random_search_rf.fit(X_train, y_train)
    rf_best_model = random_search_rf.best_estimator_
    joblib.dump(rf_best_model, config.model_storage_path / 'rf_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

rf_best_model = joblib.load(config.model_storage_path / 'rf_best_model.pkl')
param_list = ['model__n_estimators', 'model__max_depth', 'model__min_samples_split', 'model__min_samples_leaf', 'model__max_features', 'model__bootstrap']
best_params = {param: rf_best_model.get_params()[param] for param in param_list}

for param, value in best_params.items():
    print(f"{param}: {value}")

rf_model = RandomForestRegressor(
    n_estimators=best_params['model__n_estimators'],
    max_depth=best_params['model__max_depth'],
    min_samples_split=best_params['model__min_samples_split'],
    min_samples_leaf=best_params['model__min_samples_leaf'],
    max_features=best_params['model__max_features'],
    bootstrap=best_params['model__bootstrap'],
    random_state=42,
    n_jobs=-1,
    verbose=1)

Загружена модель из models
model__n_estimators: 134
model__max_depth: 11
model__min_samples_split: 6
model__min_samples_leaf: 4
model__max_features: 0.5
model__bootstrap: True


In [60]:
print('Лучшие параметры RandomForest:')
print(rf_best_model.get_params()['model'])

Лучшие параметры RandomForest:
RandomForestRegressor(max_depth=11, max_features=0.5, min_samples_leaf=4,
                      min_samples_split=6, n_estimators=134, random_state=42)


In [61]:
train_preds = rf_best_model.predict(X_train)
test_preds = rf_best_model.predict(X_test)

print("=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===")
show_metrics(train_preds, y_train)

print("----------------------------------")

print("=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===")
show_metrics(test_preds, y_test)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $10,754
RELATIVE MAE ERROR:  0.059
RMSE: $20,157
RELATIVE RMSE ERROR:  0.111
R²:   0.9154
----------------------------------
=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $16,521
RELATIVE MAE ERROR:  0.092
RMSE: $27,482
RELATIVE RMSE ERROR:  0.153
R²:   0.8476


### Подбор гиперпараметров для LightGBL

In [62]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

lgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb_model)
])

param_dist_lgb = {
    'model__n_estimators': randint(100, 500),
    'model__learning_rate': uniform(0.01, 0.2),
    'model__num_leaves': randint(20, 80),
    'model__max_depth': randint(4, 20),
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__min_child_samples': randint(5, 50),
    'model__reg_alpha': loguniform(1e-5, 1e2),
    'model__reg_lambda': loguniform(1e-5, 1e2)
}

random_search_lgb = RandomizedSearchCV(
    estimator=lgb_pipeline,
    param_distributions=param_dist_lgb,
    n_iter=100,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

if (config.model_storage_path / 'lgb_best_model.pkl').exists():
    lgb_best_model = joblib.load(config.model_storage_path / 'lgb_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    random_search_lgb.fit(X_train, y_train)
    lgb_best_model = random_search_lgb.best_estimator_
    joblib.dump(lgb_best_model, config.model_storage_path / 'lgb_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

lgb_best_model = joblib.load(config.model_storage_path / 'lgb_best_model.pkl')
param_list = ['model__n_estimators', 'model__learning_rate', 'model__num_leaves', 'model__max_depth', 'model__subsample', 'model__colsample_bytree', 'model__min_child_samples', 'model__reg_alpha', 'model__reg_lambda']
best_params = {param: lgb_best_model.get_params()[param] for param in param_list}

for param, value in best_params.items():
    print(f"{param}: {value}")

lgb_model = lgb.LGBMRegressor(
    n_estimators=best_params['model__n_estimators'],
    learning_rate=best_params['model__learning_rate'],
    num_leaves=best_params['model__num_leaves'],
    max_depth=best_params['model__max_depth'],
    subsample=best_params['model__subsample'],
    colsample_bytree=best_params['model__colsample_bytree'],
    min_child_samples=best_params['model__min_child_samples'],
    reg_alpha=best_params['model__reg_alpha'],
    reg_lambda=best_params['model__reg_lambda'],
    random_state=42,
    n_jobs=-1,
    verbose=-1)

Загружена модель из models
model__n_estimators: 403
model__learning_rate: 0.013615072723104174
model__num_leaves: 39
model__max_depth: 17
model__subsample: 0.8
model__colsample_bytree: 0.6
model__min_child_samples: 25
model__reg_alpha: 0.0036752043516458765
model__reg_lambda: 1.6188017357746958


In [63]:
print('Лучшие параметры LightGBM:')
print(lgb_best_model.get_params()['model'])

Лучшие параметры LightGBM:
LGBMRegressor(colsample_bytree=0.6,
              learning_rate=np.float64(0.013615072723104174), max_depth=17,
              min_child_samples=25, n_estimators=403, n_jobs=-1, num_leaves=39,
              random_state=42, reg_alpha=np.float64(0.0036752043516458765),
              reg_lambda=np.float64(1.6188017357746958), subsample=0.8,
              verbose=-1)


In [64]:
train_preds_lgb = lgb_best_model.predict(X_train)
test_preds_lgb = lgb_best_model.predict(X_test)

print('=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===')
show_metrics(train_preds_lgb, y_train)
print('----------------------------------')
print('=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===')
show_metrics(test_preds_lgb, y_test)

=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $8,807
RELATIVE MAE ERROR:  0.049
RMSE: $19,170
RELATIVE RMSE ERROR:  0.106
R²:   0.9299
----------------------------------
=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $15,766
RELATIVE MAE ERROR:  0.088
RMSE: $27,030
RELATIVE RMSE ERROR:  0.15
R²:   0.8680


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### Подбор гиперпараметров для XGBoost

In [65]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

param_dist_xgb = {
    'model__n_estimators': randint(100, 500),
    'model__learning_rate': uniform(0.01, 0.2),
    'model__max_depth': randint(3, 10),
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__reg_alpha': loguniform(1e-5, 1e2),
    'model__reg_lambda': loguniform(1e-5, 1e2)
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    n_iter=100,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

if (config.model_storage_path / 'xgb_best_model.pkl').exists():
    xgb_best_model = joblib.load(config.model_storage_path / 'xgb_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    random_search_xgb.fit(X_train, y_train)
    xgb_best_model = random_search_xgb.best_estimator_
    joblib.dump(xgb_best_model, config.model_storage_path / 'xgb_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

xgb_best_model = joblib.load(config.model_storage_path / 'xgb_best_model.pkl')
param_list = ['model__n_estimators', 'model__learning_rate', 'model__max_depth', 'model__subsample', 'model__colsample_bytree', 'model__reg_alpha', 'model__reg_lambda']
best_params = {param: xgb_best_model.get_params()[param] for param in param_list}

for param, value in best_params.items():
    print(f"{param}: {value}")

xgb_model = xgb.XGBRegressor(
    n_estimators=best_params['model__n_estimators'],
    learning_rate=best_params['model__learning_rate'],
    max_depth=best_params['model__max_depth'],
    subsample=best_params['model__subsample'],
    colsample_bytree=best_params['model__colsample_bytree'],    
    reg_alpha=best_params['model__reg_alpha'],
    reg_lambda=best_params['model__reg_lambda'],
    random_state=42,
    n_jobs=-1,
    verbose=-1)

Загружена модель из models
model__n_estimators: 441
model__learning_rate: 0.11105047448957144
model__max_depth: 3
model__subsample: 0.9
model__colsample_bytree: 0.6
model__reg_alpha: 0.0012118426177121782
model__reg_lambda: 5.01065164211567e-05


In [66]:
print('Лучшие параметры XGBoost:')
print(xgb_best_model.get_params()['model'])

Лучшие параметры XGBoost:
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=np.float64(0.11105047448957144), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=441, n_jobs=-1,
             num_parallel_tree=None, ...)


In [67]:
train_preds_xgb = xgb_best_model.predict(X_train)
test_preds_xgb = xgb_best_model.predict(X_test)

print('=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===')
show_metrics(train_preds_xgb, y_train)
print('----------------------------------')
print('=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===')
show_metrics(test_preds_xgb, y_test)


=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $4,069
RELATIVE MAE ERROR:  0.022
RMSE: $5,199
RELATIVE RMSE ERROR:  0.029
R²:   0.9954
----------------------------------
=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $15,778
RELATIVE MAE ERROR:  0.088
RMSE: $24,528
RELATIVE RMSE ERROR:  0.136
R²:   0.8972


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Ансамблевое обучение

### Мажоритарное голосование

In [68]:
from sklearn.ensemble import VotingRegressor

param_grid_weights = {
    'weights': [[1, 1, 1], [1, 2, 1], [2, 1, 1], [1, 1, 2], [2, 2, 1], [1, 2, 2], [1, 2, 3], [1, 3, 2], [1, 2, 4]]
}

voiting_regressor = VotingRegressor(
    estimators=[
        ('rf', rf_best_model),
        ('lgb', lgb_best_model),
        ('xgb', xgb_best_model)
    ]
)

grid_weights = GridSearchCV(
    voiting_regressor,
    param_grid_weights,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

if (config.model_storage_path / 'voting_best_model.pkl').exists():
    voting_best_model = joblib.load(config.model_storage_path / 'voting_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    grid_weights.fit(X_train, y_train)
    voting_best_model = grid_weights.best_estimator_
    joblib.dump(voting_best_model, config.model_storage_path / 'voting_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

voting_model = joblib.load(config.model_storage_path / 'voting_best_model.pkl')
print(f"Лучшие веса: {voting_model.get_params()['weights']}")

voting_train_preds = voting_model.predict(X_train)
voting_test_preds = voting_model.predict(X_test)

Загружена модель из models
Лучшие веса: [1, 2, 3]


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.v

In [69]:
print('=== МЕТРИКИ VotingRegressor МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===')
show_metrics(voting_train_preds, y_train)
print('----------------------------------')
print('=== МЕТРИКИ VotingRegressor МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===')
show_metrics(voting_test_preds, y_test)

=== МЕТРИКИ VotingRegressor МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $6,295
RELATIVE MAE ERROR:  0.035
RMSE: $10,917
RELATIVE RMSE ERROR:  0.06
R²:   0.9780
----------------------------------
=== МЕТРИКИ VotingRegressor МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $15,108
RELATIVE MAE ERROR:  0.084
RMSE: $24,725
RELATIVE RMSE ERROR:  0.137
R²:   0.8896


## Стекинг

In [70]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso,LinearRegression
from sklearn.neighbors import KNeighborsRegressor

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', rf_model)
])

lgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', lgb_model)
])

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', xgb_model)
])

ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', Ridge(alpha=1.0))
])

knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', KNeighborsRegressor(n_neighbors=10))
])

base_models = [
    ('rf', rf_pipeline),
    ('lgb', lgb_pipeline),
    ('xgb', xgb_pipeline),
    ('ridge', ridge_pipeline),
    ('knn', knn_pipeline)
]

stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1,
    verbose=1
)

to_relearn = True

if (config.model_storage_path / 'stacking_model.pkl').exists() and to_relearn == False:
    stacking_model = joblib.load(config.model_storage_path / 'stacking_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    stacking.fit(X_train, y_train)
    joblib.dump(stacking, config.model_storage_path / 'stacking_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

stacking_model = joblib.load(config.model_storage_path / 'stacking_model.pkl')

stacking_train_preds = stacking_model.predict(X_train)
stacking_test_preds = stacking_model.predict(X_test)

Модель обучена и сохранена в models
(1022, 35)
(1022, 35)


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 134 out of 134 | elapsed:    0.0s finished
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


(1022, 35)
(1022, 35)
(1022, 35)
(438, 35)


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 134 out of 134 | elapsed:    0.0s finished
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-hous

(438, 35)
(438, 35)
(438, 35)
(438, 35)


In [71]:
print('=== МЕТРИКИ StackingRegressor МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===')
show_metrics(stacking_train_preds, y_train)
print('----------------------------------')
print('=== МЕТРИКИ StackingRegressor МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===')
show_metrics(stacking_test_preds, y_test)

=== МЕТРИКИ StackingRegressor МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $9,459
RELATIVE MAE ERROR:  0.052
RMSE: $20,362
RELATIVE RMSE ERROR:  0.112
R²:   0.9251
----------------------------------
=== МЕТРИКИ StackingRegressor МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $15,664
RELATIVE MAE ERROR:  0.087
RMSE: $26,717
RELATIVE RMSE ERROR:  0.148
R²:   0.8777


#### Далее модель сгенерированная нейронкой

In [72]:
xgb_final = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_alpha=0.01,
    reg_lambda=1.0    
)

xgb_pipeline_final =Pipeline([
    ('preprocessor', preprocessor),    
    ('model', xgb_final)
])

xgb_pipeline_final.fit(X_train, y_train)
pred = xgb_pipeline_final.predict(X_test)

print(f"Test R²: {r2_score(y_test, pred):.4f}")

(1022, 35)
(438, 35)
Test R²: 0.9224


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
